# MySQL Enterprise with KMIP

`./demo.sh` includes this stage. It reuses `mysql/mysql-enterprise-0`; an empty installation uses MySQL Enterprise 8.3.0, with the `aarch64` variant on ARM64. The Oracle registry credentials and the initial password are in `.env`. Registry access must be enabled beforehand in the Oracle account.

Vault serves KMIP at `vault-kmip.vault.svc.cluster.local:5696`. MySQL loads `keyring_okv`; the Lumen tables use `ENCRYPTION='Y'`. Existing client certificates are preserved.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/mysql.sh


## Check the plugin and encryption

The query uses the password from the pod environment and does not display it. The tables appear after running the database roles stage.


In [ ]:
%%bash
set -euo pipefail
kubectl --context workshop -n mysql exec -i mysql-enterprise-0 -c mysql -- \
  sh -c 'MYSQL_PWD="$MYSQL_ROOT_PASSWORD" exec mysql -uroot --batch' <<'SQL'
SELECT PLUGIN_NAME, PLUGIN_STATUS FROM INFORMATION_SCHEMA.PLUGINS WHERE PLUGIN_NAME='keyring_okv';
SELECT NAME, ENCRYPTION FROM INFORMATION_SCHEMA.INNODB_TABLESPACES WHERE NAME LIKE 'scripting/%';
SQL


The automated demo does not change the KMIP port to simulate failures, does not rotate the InnoDB master key and does not reinstall UDF functions. Those were manual exercises from the previous lab; they are not Lumen requirements. Nor does it restart MySQL on every run or delete its PVCs.
